In [13]:
import os
import dotenv
dotenv.load_dotenv()
my_api_key = os.getenv('pnu_RPuCabmuMJWNToFgTGctGM1viJrRxo1em6DI')

In [ ]:
%%writefile wmata_pull.py
from prefect import flow, task
from datetime import datetime
import requests
import json
import os
import time

api_key = "4b20a99005d64434999fae74f7ba3f7c"
url = "http://api.wmata.com/StationPrediction.svc/json/GetPrediction/All"
output_file = "wmata_data.json"
headers = {"api_key": api_key}

@task
def fetch_api_data():
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        pass
    else:
        print(f"Error: {response.status_code}")

    data = response.json()
    data = {
        'Time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        **data
    }
    return data

@task
def append_to_json_file(new_data):
    if new_data is None:
        return
    if os.path.exists(output_file):
        with open(output_file, "r") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                data = []
    else:
        data = []
    data.append(new_data)
    with open(output_file, "w") as f:
        json.dump(data, f, indent=4)

@flow
def scheduled_api_flow():
    data = fetch_api_data()
    append_to_json_file(data)


Writing wmata_pull.py


dict

In [11]:
for key1, value1 in list(data.items()):
    for value2 in value1:
        print(f"{value2}")

2
0
2
5
-
0
4
-
1
2
 
1
4
:
2
9
:
5
2
{'Car': '6', 'Destination': 'Glenmont', 'DestinationCode': 'B11', 'DestinationName': 'Glenmont', 'Group': '1', 'Line': 'RD', 'LocationCode': 'A01', 'LocationName': 'Metro Center', 'Min': 'BRD'}
{'Car': '8', 'Destination': 'Glenmont', 'DestinationCode': 'B11', 'DestinationName': 'Glenmont', 'Group': '1', 'Line': 'RD', 'LocationCode': 'A04', 'LocationName': 'Woodley Park-Zoo/Adams Morgan', 'Min': 'BRD'}
{'Car': '8', 'Destination': 'Shady Grv', 'DestinationCode': None, 'DestinationName': 'Shady Grv', 'Group': '2', 'Line': 'RD', 'LocationCode': 'A08', 'LocationName': 'Friendship Heights', 'Min': 'BRD'}
{'Car': '8', 'Destination': 'Glenmont', 'DestinationCode': 'B11', 'DestinationName': 'Glenmont', 'Group': '1', 'Line': 'RD', 'LocationCode': 'A13', 'LocationName': 'Twinbrook', 'Min': 'BRD'}
{'Car': '8', 'Destination': 'Shady Grv', 'DestinationCode': None, 'DestinationName': 'Shady Grv', 'Group': '2', 'Line': 'RD', 'LocationCode': 'A13', 'LocationName': 

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime
import requests
from collections import defaultdict
from typing import List
import pandas as pd
import time

class TrainEntry(BaseModel):
    Car: Optional[str]
    Destination: str
    DestinationCode: Optional[str]
    DestinationName: str
    Group: str
    Line: str
    LocationCode: Optional[str]
    LocationName: str
    Min: str
    CallTime: datetime
    Peak: bool
    LeftYet: bool = False
    IsNext: bool = False
    BRD_Time: Optional[datetime] = None
    TimeToBRD: Optional[float] = None
    Diff: Optional[float] = None

def is_peak_hour(dt: datetime) -> bool:
    return dt.weekday() < 5 and dt.hour >= 5 and dt.hour < 21 or (dt.hour == 21 and dt.minute <= 30)


def get_batch() -> list[TrainEntry]:
    api_key = "4b20a99005d64434999fae74f7ba3f7c"
    url = "http://api.wmata.com/StationPrediction.svc/json/GetPrediction/All"
    output_file = "wmata_data.json"
    headers = {"api_key": api_key}
    response = requests.get(url, headers=headers)
    data = response.json()
    call_time = datetime.now()
    peak = is_peak_hour(call_time)

    trains = []
    for train in data.get("Trains", []):
        if train.get("Min") == "---":
            continue
        entry = TrainEntry(
            **train,
            CallTime=call_time,
            Peak=peak
        )
        trains.append(entry)
    #print(trains)
    return trains

def transform_batch(current_batch: List[TrainEntry], all_past_batches: List[TrainEntry], last_batch: List[TrainEntry]):
    from collections import defaultdict
    from datetime import datetime

    # Group current batch by Line, Location, Destination
    groups = defaultdict(list)
    for t in current_batch:
        key = (t.Line, t.LocationName, t.Destination)
        groups[key].append(t)

    # Step 1: Compute IsNext for current batch
    for key, trains in groups.items():
        not_left = [t for t in trains if not t.LeftYet]
        digit_mins = [int(t.Min) for t in not_left if str(t.Min).isdigit()]
        if digit_mins:
            min_val = min(digit_mins)
            for t in not_left:
                if t.Min.isdigit() and int(t.Min) == min_val:
                    t.IsNext = True

    # Step 2: Detect new arrivals
    arriving_keys = {
        (t.Line, t.LocationName, t.Destination)
        for t in current_batch
        if t.Min in {"ARR", "BRD"}
    }
    previously_arriving_keys = {
        (t.Line, t.LocationName, t.Destination)
        for t in last_batch
        if t.Min in {"ARR", "BRD"}
    }
    new_arrivals = arriving_keys - previously_arriving_keys

    # Step 3: For each new arrival, update IsNext and LeftYet retroactively
    arrival_time = current_batch[0].CallTime  # assuming consistent call time in batch
    for key in new_arrivals:
        # First retroactively update IsNext on all past trains that were next
        for t in all_past_batches:
            if (
                (t.Line, t.LocationName, t.Destination) == key and
                not t.LeftYet and
                str(t.Min).isdigit()
            ):
                # Check if it was the next train in its call group
                min_val = min(
                    int(x.Min) for x in all_past_batches
                    if x.CallTime == t.CallTime and
                    (x.Line, x.LocationName, x.Destination) == key and
                    not x.LeftYet and
                    str(x.Min).isdigit()
                )
                if int(t.Min) == min_val:
                    t.IsNext = True

        # Step 4: Mark IsNext trains as LeftYet and update timings
        for t in all_past_batches:
            if (t.Line, t.LocationName, t.Destination) == key and t.IsNext and not t.LeftYet:
                t.LeftYet = True
                t.BRD_Time = arrival_time
                delta = (arrival_time - t.CallTime).total_seconds() / 60.0
                t.TimeToBRD = delta
                if str(t.Min).isdigit():
                    t.Diff = delta - int(t.Min)

    # Step 5: Recompute IsNext globally after updates
    # Reset all IsNext first
    for t in all_past_batches + current_batch:
        t.IsNext = False

    combined = all_past_batches + current_batch
    call_groups = defaultdict(list)
    for t in combined:
        if not t.LeftYet and str(t.Min).isdigit():
            key = (t.CallTime, t.Line, t.LocationName, t.Destination)
            call_groups[key].append(t)

    for key, trains in call_groups.items():
        min_val = min(int(t.Min) for t in trains)
        for t in trains:
            if int(t.Min) == min_val:
                t.IsNext = True



In [ ]:
all_trains = []
last_batch = []
count = 1

for _ in range(60):
    print("Call", count, ":", datetime.now())
    count+=1
    current_batch = get_batch()
    transform_batch(current_batch, all_trains, last_batch)
    all_trains.extend(current_batch)
    last_batch = current_batch
    time.sleep(30)

df = pd.DataFrame([t.dict() for t in all_trains])

Call 1 : 2025-04-30 15:27:58.517281
Call 2 : 2025-04-30 15:28:28.662068
Call 3 : 2025-04-30 15:28:59.170547
Call 4 : 2025-04-30 15:29:29.449530
Call 5 : 2025-04-30 15:29:59.655171
Call 6 : 2025-04-30 15:30:29.878879
Call 7 : 2025-04-30 15:31:00.293825
Call 8 : 2025-04-30 15:31:30.854243
Call 9 : 2025-04-30 15:32:01.201119
Call 10 : 2025-04-30 15:32:32.259647
Call 11 : 2025-04-30 15:33:02.901989
Call 12 : 2025-04-30 15:33:33.363499
Call 13 : 2025-04-30 15:34:03.802052
Call 14 : 2025-04-30 15:34:34.218499
Call 15 : 2025-04-30 15:35:04.881011
Call 16 : 2025-04-30 15:35:35.397906
Call 17 : 2025-04-30 15:36:06.117976
Call 18 : 2025-04-30 15:36:36.780243
Call 19 : 2025-04-30 15:37:07.401652
Call 20 : 2025-04-30 15:37:38.152123
Call 21 : 2025-04-30 15:38:09.215890
Call 22 : 2025-04-30 15:38:39.877979
Call 23 : 2025-04-30 15:39:10.904750
Call 24 : 2025-04-30 15:39:41.810463
Call 25 : 2025-04-30 15:40:12.611777
Call 26 : 2025-04-30 15:40:43.621006
Call 27 : 2025-04-30 15:41:14.579376
Call 28 : 

/var/folders/z5/l6g0391s0qg3vsbnvl7y81n80000gn/T/ipykernel_10351/41810299.py:17: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  df = pd.DataFrame([t.dict() for t in all_trains])


In [63]:
%%writefile pipeline.py
from pydantic import BaseModel
from typing import Optional, List
from datetime import datetime
import requests
import pandas as pd
from collections import defaultdict
from prefect import flow, task
import pickle
from pathlib import Path

class TrainEntry(BaseModel):
    Car: Optional[str]
    Destination: str
    DestinationCode: Optional[str]
    DestinationName: str
    Group: str
    Line: str
    LocationCode: Optional[str]
    LocationName: str
    Min: str
    CallTime: datetime
    Peak: bool
    LeftYet: bool = False
    IsNext: bool = False
    BRD_Time: Optional[datetime] = None
    TimeToBRD: Optional[float] = None
    Diff: Optional[float] = None

def is_peak_hour(dt: datetime) -> bool:
    return dt.weekday() < 5 and dt.hour >= 5 and dt.hour < 21 or (dt.hour == 21 and dt.minute <= 30)

@task
def get_batch() -> List[TrainEntry]:
    api_key = "4b20a99005d64434999fae74f7ba3f7c"
    url = "http://api.wmata.com/StationPrediction.svc/json/GetPrediction/All"
    headers = {"api_key": api_key}
    response = requests.get(url, headers=headers)
    data = response.json()
    call_time = datetime.now()
    peak = is_peak_hour(call_time)

    trains = []
    for train in data.get("Trains", []):
        if train.get("Min") == "---":
            continue
        entry = TrainEntry(
            **train,
            CallTime=call_time,
            Peak=peak
        )
        trains.append(entry)
    return trains

@task
def load_state():
    path = Path("train_state.pkl")
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    return [], []

@task
def save_state(all_trains, last_batch):
    with open("train_state.pkl", "wb") as f:
        pickle.dump((all_trains, last_batch), f)

@task
def transform_batch(current_batch: List[TrainEntry], all_past_batches: List[TrainEntry], last_batch: List[TrainEntry]):
    groups = defaultdict(list)
    for t in current_batch:
        key = (t.Line, t.LocationName, t.Destination)
        groups[key].append(t)

    for key, trains in groups.items():
        not_left = [t for t in trains if not t.LeftYet]
        digit_mins = [int(t.Min) for t in not_left if str(t.Min).isdigit()]
        if digit_mins:
            min_val = min(digit_mins)
            for t in not_left:
                if t.Min.isdigit() and int(t.Min) == min_val:
                    t.IsNext = True

    arriving_keys = {
        (t.Line, t.LocationName, t.Destination)
        for t in current_batch
        if t.Min in {"ARR", "BRD"}
    }
    previously_arriving_keys = {
        (t.Line, t.LocationName, t.Destination)
        for t in last_batch
        if t.Min in {"ARR", "BRD"}
    }
    new_arrivals = arriving_keys - previously_arriving_keys

    arrival_time = current_batch[0].CallTime
    for key in new_arrivals:
        for t in all_past_batches:
            if (
                (t.Line, t.LocationName, t.Destination) == key and
                not t.LeftYet and
                str(t.Min).isdigit()
            ):
                min_val = min(
                    int(x.Min) for x in all_past_batches
                    if x.CallTime == t.CallTime and
                    (x.Line, x.LocationName, x.Destination) == key and
                    not x.LeftYet and
                    str(x.Min).isdigit()
                )
                if int(t.Min) == min_val:
                    t.IsNext = True

        for t in all_past_batches:
            if (t.Line, t.LocationName, t.Destination) == key and t.IsNext and not t.LeftYet:
                t.LeftYet = True
                t.BRD_Time = arrival_time
                delta = (arrival_time - t.CallTime).total_seconds() / 60.0
                t.TimeToBRD = delta
                if str(t.Min).isdigit():
                    t.Diff = delta - int(t.Min)

    for t in all_past_batches + current_batch:
        t.IsNext = False

    combined = all_past_batches + current_batch
    call_groups = defaultdict(list)
    for t in combined:
        if not t.LeftYet and str(t.Min).isdigit():
            key = (t.CallTime, t.Line, t.LocationName, t.Destination)
            call_groups[key].append(t)

    for key, trains in call_groups.items():
        min_val = min(int(t.Min) for t in trains)
        for t in trains:
            if int(t.Min) == min_val:
                t.IsNext = True

    return current_batch

@flow
def scheduled_train_flow():
    all_trains, last_batch = load_state()
    current_batch = get_batch()
    updated_current_batch = transform_batch(current_batch, all_trains, last_batch)
    all_trains.extend(updated_current_batch)
    last_batch = updated_current_batch
    save_state(all_trains, last_batch)
    print("Batch processed at:", datetime.now())

if __name__ == "__main__":
    scheduled_train_flow()

Writing pipeline.py


In [59]:
df.to_csv('30min5.csv', index=False)

In [131]:
%%writefile new_pipeline.py
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime
import requests
from collections import defaultdict
from typing import Optional, List
import pandas as pd
import time
from prefect import flow, task
import os
import json

class TrainEntry(BaseModel):
    Car: Optional[str]
    Destination: str
    DestinationCode: Optional[str]
    DestinationName: str
    Group: str
    Line: str
    LocationCode: Optional[str]
    LocationName: str
    Min: str
    CallTime: datetime
    Peak: bool
    LeftYet: bool = False
    IsNext: bool = False
    BRD_Time: Optional[datetime] = None
    TimeToBRD: Optional[float] = None
    Diff: Optional[float] = None

def is_peak_hour(dt: datetime) -> bool:
    return dt.weekday() < 5 and dt.hour >= 5 and dt.hour < 21 or (dt.hour == 21 and dt.minute <= 30)


def get_batch() -> list[TrainEntry]:
    api_key = "4b20a99005d64434999fae74f7ba3f7c"
    url = "http://api.wmata.com/StationPrediction.svc/json/GetPrediction/All"
    output_file = "wmata_data.json"
    headers = {"api_key": api_key}
    response = requests.get(url, headers=headers)
    data = response.json()
    call_time = datetime.now()
    peak = is_peak_hour(call_time)

    trains = []
    for train in data.get("Trains", []):
        if train.get("Min") == "---":
            continue
        entry = TrainEntry(
            **train,
            CallTime=call_time,
            Peak=peak
        )
        trains.append(entry)
    #print(trains)
    return trains

def transform_batch(current_batch: List[TrainEntry], all_past_batches: List[TrainEntry], last_batch: List[TrainEntry]):
    from collections import defaultdict
    from datetime import datetime

    # Group current batch by Line, Location, Destination
    groups = defaultdict(list)
    for t in current_batch:
        key = (t.Line, t.LocationName, t.Destination)
        groups[key].append(t)

    # Step 1: Compute IsNext for current batch
    for key, trains in groups.items():
        not_left = [t for t in trains if not t.LeftYet]
        digit_mins = [int(t.Min) for t in not_left if str(t.Min).isdigit()]
        if digit_mins:
            min_val = min(digit_mins)
            for t in not_left:
                if t.Min.isdigit() and int(t.Min) == min_val:
                    t.IsNext = True

    # Step 2: Detect new arrivals
    arriving_keys = {
        (t.Line, t.LocationName, t.Destination)
        for t in current_batch
        if t.Min in {"ARR", "BRD"}
    }
    previously_arriving_keys = {
        (t.Line, t.LocationName, t.Destination)
        for t in last_batch
        if t.Min in {"ARR", "BRD"}
    }
    new_arrivals = arriving_keys - previously_arriving_keys

    # Step 3: For each new arrival, update IsNext and LeftYet retroactively
    arrival_time = current_batch[0].CallTime  # assuming consistent call time in batch
    for key in new_arrivals:
        # First retroactively update IsNext on all past trains that were next
        for t in all_past_batches:
            if (
                (t.Line, t.LocationName, t.Destination) == key and
                not t.LeftYet and
                str(t.Min).isdigit()
            ):
                # Check if it was the next train in its call group
                min_val = min(
                    int(x.Min) for x in all_past_batches
                    if x.CallTime == t.CallTime and
                    (x.Line, x.LocationName, x.Destination) == key and
                    not x.LeftYet and
                    str(x.Min).isdigit()
                )
                if int(t.Min) == min_val:
                    t.IsNext = True

        # Step 4: Mark IsNext trains as LeftYet and update timings
        for t in all_past_batches:
            if (t.Line, t.LocationName, t.Destination) == key and t.IsNext and not t.LeftYet:
                t.LeftYet = True
                t.BRD_Time = arrival_time
                delta = (arrival_time - t.CallTime).total_seconds() / 60.0
                t.TimeToBRD = delta
                if str(t.Min).isdigit():
                    t.Diff = delta - int(t.Min)

    # Step 5: Recompute IsNext globally after updates
    # Reset all IsNext first
    for t in all_past_batches + current_batch:
        t.IsNext = False

    combined = all_past_batches + current_batch
    call_groups = defaultdict(list)
    for t in combined:
        if not t.LeftYet and str(t.Min).isdigit():
            key = (t.CallTime, t.Line, t.LocationName, t.Destination)
            call_groups[key].append(t)

    for key, trains in call_groups.items():
        min_val = min(int(t.Min) for t in trains)
        for t in trains:
            if int(t.Min) == min_val:
                t.IsNext = True

def save_to_json(data, filename):
    with open(filename, 'w') as f:
        json.dump([t.dict() for t in data], f, indent=4)

def load_from_json(filename) -> List[TrainEntry]:
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            data = json.load(f)
            return [TrainEntry.from_dict(d) for d in data]
    else:
        return []

@task
def load_state():
    return load_from_json("all_trains.json")

@task
def save_state(train_list):
    save_to_json(train_list, "all_trains.json")

@task
def load_last_batch():
    return load_from_json("last_batch.json")

@task
def save_last_batch(last_batch):
    save_to_json(last_batch, "last_batch.json")

Overwriting new_pipeline.py


In [ ]:
%%writefile serve.py
from datetime import timedelta
from prefect import flow
from new_pipeline import get_batch, transform_batch, load_state, save_state, load_last_batch, save_last_batch
import pandas as pd
from datetime import datetime
import json

CSV_FILE = "train_data.csv"

@flow(log_prints=True)
def scheduled_train_flow():
    print("Running batch at", datetime.now())
    
    all_trains = load_state()
    last_batch = load_last_batch()
    current_batch = get_batch()
    transform_batch(current_batch, all_trains, last_batch)

    save_state(all_trains)
    save_last_batch(last_batch)

    # Save the updated data to CSV after every batch
    df = pd.DataFrame([t.dict() for t in all_trains])
    df.to_csv(CSV_FILE, index=False)
    print(f"Data saved to {CSV_FILE}, total records: {len(df)}")

if __name__ == "__main__":
    scheduled_train_flow.serve(
        name="train-data-pipeline",
        interval=timedelta(seconds=30)
    )

Overwriting serve.py


In [170]:
%%writefile pipeline.py
import requests
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
from pandas import Timestamp
from prefect import flow, task
import os

# Helper function to check if it's peak hour
def is_peak_hour(dt: datetime) -> bool:
    return dt.weekday() < 5 and dt.hour >= 5 and dt.hour < 21 or (dt.hour == 21 and dt.minute <= 30)

# Fetch data from API and convert it to a DataFrame
def get_batch() -> pd.DataFrame:
    api_key = "4b20a99005d64434999fae74f7ba3f7c"
    url = "http://api.wmata.com/StationPrediction.svc/json/GetPrediction/All"
    headers = {"api_key": api_key}
    response = requests.get(url, headers=headers)
    data = response.json()
    
    call_time = datetime.now()
    peak = is_peak_hour(call_time)
    
    # Prepare a list of dictionaries for each train entry
    trains = []
    for train in data.get("Trains", []):
        if train.get("Min") == "---":
            continue
        
        # Create the full dictionary for each entry
        entry = {
            "Car": train.get("Car"),
            "Destination": train.get("Destination"),
            "DestinationCode": train.get("DestinationCode"),
            "DestinationName": train.get("DestinationName"),
            "Group": train.get("Group"),
            "Line": train.get("Line"),
            "LocationCode": train.get("LocationCode"),
            "LocationName": train.get("LocationName"),
            "Min": train.get("Min"),
            "CallTime": call_time,
            "Peak": peak,
            "LeftYet": False,
            "IsNext": False,
            "BRD_Time": None,
            "TimeToBRD": None,
            "Diff": None
        }
        
        # Append to the trains list
        trains.append(entry)
    
    # Convert to a DataFrame and return
    return pd.DataFrame(trains)

@task
def fetch_data():
    api_key = "4b20a99005d64434999fae74f7ba3f7c"
    url = "http://api.wmata.com/StationPrediction.svc/json/GetPrediction/All"
    headers = {"api_key": api_key}
    response = requests.get(url, headers=headers)
    data = response.json()
    
    call_time = datetime.now()
    peak = is_peak_hour(call_time)
    
    # Prepare a list of dictionaries for each train entry
    trains = []
    for train in data.get("Trains", []):
        if train.get("Min") == "---":
            continue
        
        # Create the full dictionary for each entry
        entry = {
            "Car": train.get("Car"),
            "Destination": train.get("Destination"),
            "DestinationCode": train.get("DestinationCode"),
            "DestinationName": train.get("DestinationName"),
            "Group": train.get("Group"),
            "Line": train.get("Line"),
            "LocationCode": train.get("LocationCode"),
            "LocationName": train.get("LocationName"),
            "Min": train.get("Min"),
            "CallTime": call_time,
            "Peak": peak,
            "LeftYet": False,
            "IsNext": False,
            "BRD_Time": None,
            "TimeToBRD": None,
            "Diff": None
        }
        
        # Append to the trains list
        trains.append(entry)
    
    # Convert to a DataFrame and return
    return pd.DataFrame(trains)

@task
def load_past_data():
    if os.path.exists("all_trains2.csv"):
        all_trains = pd.read_csv("all_trains2.csv", parse_dates=["CallTime", "BRD_Time"])
    else:
        all_trains = pd.DataFrame()

    if os.path.exists("last_batch2.csv"):
        last_batch = pd.read_csv("last_batch2.csv", parse_dates=["CallTime"])
    else:
        last_batch = pd.DataFrame()

    return all_trains, last_batch

@task
def save_data(all_trains: pd.DataFrame, current_batch: pd.DataFrame):
    all_trains.to_csv("all_trains2.csv", index=False)
    current_batch.to_csv("last_batch2.csv", index=False)

# Transform batch to update IsNext, LeftYet, etc.
def transform_batch(current_batch: pd.DataFrame, all_trains: pd.DataFrame, last_batch: pd.DataFrame) -> pd.DataFrame:
    from collections import defaultdict
    from datetime import datetime

    # Ensure 'IsNext', 'LeftYet', 'BRD_Time', 'TimeToBRD', and 'Diff' columns exist
    for col in ['IsNext', 'LeftYet', 'BRD_Time', 'TimeToBRD', 'Diff']:
        for df in [current_batch, all_trains]:
            if col not in df.columns:
                df[col] = None

    current_batch['IsNext'] = False
    current_batch['LeftYet'] = False

    # Step 1: Compute IsNext for current batch
    group_keys = ['Line', 'LocationName', 'Destination']
    
    # Safe conversion function for Min values
    def safe_to_int(value):
        try:
            return int(value)
        except ValueError:
            return None

    for key, group in current_batch.groupby(group_keys):
        not_left = group[~group['LeftYet']]

        # Filter out non-numeric Min values (only consider numeric)
        digit_mins = not_left[not_left['Min'].apply(lambda x: safe_to_int(x) is not None)]

        if not digit_mins.empty:
            min_val = digit_mins['Min'].apply(safe_to_int).min()
            idx = digit_mins[digit_mins['Min'].apply(safe_to_int) == min_val].index
            current_batch.loc[idx, 'IsNext'] = True

    # Step 2: Detect new arrivals
    arrival_keys = set(map(tuple, current_batch[current_batch['Min'].isin(['ARR', 'BRD'])][group_keys].values))
    previous_arrival_keys = set(map(tuple, last_batch[last_batch['Min'].isin(['ARR', 'BRD'])][group_keys].values))
    new_arrivals = arrival_keys - previous_arrival_keys

    # Use CallTime from batch
    arrival_time = current_batch['CallTime'].iloc[0] if not current_batch.empty else datetime.now()

    # Step 3: For each new arrival, update IsNext and LeftYet retroactively
    for key in new_arrivals:
        mask_past = (
            (all_trains['Line'] == key[0]) &
            (all_trains['LocationName'] == key[1]) &
            (all_trains['Destination'] == key[2]) &
            (~all_trains['LeftYet']) &
            (all_trains['Min'].apply(lambda x: safe_to_int(x) is not None))
        )

        for ct in all_trains[mask_past]['CallTime'].unique():
            ct_mask = mask_past & (all_trains['CallTime'] == ct)
            if not all_trains[ct_mask].empty:
                digit_ct_mask = ct_mask & all_trains['Min'].apply(lambda x: safe_to_int(x) is not None)

                if not all_trains[digit_ct_mask].empty:
                    min_val = all_trains.loc[digit_ct_mask, 'Min'].apply(safe_to_int).min()
                    is_next_mask = digit_ct_mask & (all_trains['Min'].apply(safe_to_int) == min_val)
                    all_trains.loc[is_next_mask, 'IsNext'] = True

        retro_mask = (
            (all_trains['Line'] == key[0]) &
            (all_trains['LocationName'] == key[1]) &
            (all_trains['Destination'] == key[2]) &
            (all_trains['IsNext']) &
            (~all_trains['LeftYet'])
        )

        # Update LeftYet and other columns
        all_trains.loc[retro_mask, 'LeftYet'] = True
        all_trains.loc[retro_mask, 'BRD_Time'] = arrival_time

        # Only update TimeToBRD and Diff if 'Min' is numeric
        valid_min_mask = retro_mask & all_trains['Min'].apply(lambda x: safe_to_int(x) is not None)
        all_trains.loc[valid_min_mask, 'TimeToBRD'] = (
            (arrival_time - all_trains.loc[valid_min_mask, 'CallTime']).dt.total_seconds() / 60
        )
        all_trains.loc[valid_min_mask, 'Diff'] = (
            all_trains.loc[valid_min_mask, 'TimeToBRD'] -
            all_trains.loc[valid_min_mask, 'Min'].apply(safe_to_int)
        )

    # Step 4: Final IsNext recompute
    all_trains['IsNext'] = False
    combined = pd.concat([all_trains, current_batch], ignore_index=True)

    # Filter valid Min rows before computing IsNext
    valid_min_combined = combined[combined['Min'].apply(lambda x: safe_to_int(x) is not None)]
    call_groups = valid_min_combined.groupby(['CallTime', 'Line', 'LocationName', 'Destination'])

    for _, group in call_groups:
        min_val = group['Min'].apply(safe_to_int).min()
        idx_to_update = group[group['Min'].apply(safe_to_int) == min_val].index
        combined.loc[idx_to_update, 'IsNext'] = True

    return combined

@flow
def main_pipeline():
    current_batch = fetch_data()
    all_trains, last_batch = load_past_data()
    updated_all_trains = transform_batch(current_batch, all_trains, last_batch)
    save_data(updated_all_trains, current_batch)

'''
# Main ETL process
def main():
    all_trains = pd.DataFrame()  # Empty dataframe to store all data
    last_batch = pd.DataFrame()  # Empty dataframe to store the last batch of data
    
    # Run for a certain number of iterations (or until you decide to stop)
    for _ in range(60):
        current_batch = get_batch()
        
        # If there is a last batch, transform the data
        if not last_batch.empty:
            transform_batch(current_batch, all_trains, last_batch)
        
        # Append current batch to all_trains
        all_trains = pd.concat([all_trains, current_batch])
        
        # Save all_trains to CSV after each batch
        all_trains.to_csv("train_data.csv", index=False)
        
        # Update last_batch for the next iteration
        last_batch = current_batch
        
        # Sleep for 30 seconds before getting the next batch
        time.sleep(30)

if __name__ == "__main__":
    main()
'''

Overwriting pipeline.py


In [171]:
%%writefile serve.py
import time
from datetime import datetime, timedelta
from pipeline import main_pipeline

def run_schedule_for_duration(interval_seconds=30, duration_minutes=30):
    end_time = datetime.now() + timedelta(minutes=duration_minutes)
    run_count = 0

    while datetime.now() < end_time:
        loop_start = datetime.now()
        print(f"\n[{loop_start}] Starting pipeline iteration {run_count + 1}")
        
        # Run the pipeline
        main_pipeline()
        run_count += 1

        # Calculate elapsed and sleep time
        elapsed = (datetime.now() - loop_start).total_seconds()
        sleep_time = max(0, interval_seconds - elapsed)
        if sleep_time > 0:
            print(f"[{datetime.now()}] Sleeping for {sleep_time:.2f} seconds...")
            time.sleep(sleep_time)

    print(f"\n[{datetime.now()}] Completed {run_count} iterations in {duration_minutes} minutes.")

if __name__ == "__main__":
    run_schedule_for_duration()

Overwriting serve.py


In [ ]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

st.set_page_config(layout="wide")
df = pd.read_csv("all_trains1.csv")
#df2 = pd.read_csv("all_trains2.csv")
#df3 = pd.read_csv("all_trains3.csv")

#df = pd.concat([df1, df2, df3])

df["Diff"] = df["Diff"].round()
df = df[~df["Line"].isin(["--", "No"])]

st.sidebar.header("Filter Options")


line_options = sorted(df["Line"].dropna().unique())
location_options = sorted(df["LocationName"].dropna().unique())
min_range = df["Min"].apply(lambda x: int(x) if str(x).isdigit() else None).dropna()

selected_line = st.sidebar.selectbox("Select Rail Line", ["All"] + line_options)
selected_location = st.sidebar.selectbox("Select Station Location", ["All"] + location_options)
min_slider = st.sidebar.slider("Estimated Minutes Away", int(min_range.min()), int(min_range.max()), (int(min_range.min()), int(min_range.max())))

peak_filter = st.sidebar.radio("Peak Hours?", options=["All", "Peak Hours", "Off Peak"])
filtered_df = df.copy()

plot_color = "#ff4b4b"
title_color = "All Lines"
title_location = "All Stations"
if selected_line != "All":
    filtered_df = filtered_df[filtered_df["Line"] == selected_line]
    if selected_line=="BL":
        plot_color = "#0279c1"
        title_color = "Blue Line"
    elif selected_line=="OR":
        plot_color = "#f78e1d"
        title_color = "Orange Line"
    elif selected_line=="SV":
        plot_color = "#d8d8d8"
        title_color = "Silver Line"
    elif selected_line=="YL":
        plot_color = "#ffdd04"
        title_color = "Yellow Line"
    elif selected_line=="GR":
        plot_color = "#01a94f"
        title_color = "Green Line"
    elif selected_line=="RD":
        plot_color = "#ee4135"
        title_color = "Red Line"
if selected_location != "All":
    filtered_df = filtered_df[filtered_df["LocationName"] == selected_location]
    title_location = selected_location

filtered_df = filtered_df[filtered_df["Min"].apply(lambda x: str(x).isdigit())]
filtered_df["Min"] = filtered_df["Min"].astype(int)
filtered_df = filtered_df[
    (filtered_df["Min"] >= min_slider[0]) & (filtered_df["Min"] <= min_slider[1])
]

if peak_filter == "Yes":
    filtered_df = filtered_df[filtered_df["Peak"] == True]
elif peak_filter == "No":
    filtered_df = filtered_df[filtered_df["Peak"] == False]

arrivals_df = filtered_df.dropna(subset=["BRD_Time"]).copy()
arrivals_df["BRD_Time"] = pd.to_datetime(arrivals_df["BRD_Time"])
arrivals_df = arrivals_df.drop_duplicates(
    subset=["Line", "LocationName", "Destination", "BRD_Time"]
)

arrivals_df = arrivals_df.sort_values(by=["Line", "LocationName", "Destination", "BRD_Time"])

arrivals_df["GapMin"] = (
    arrivals_df.groupby(["Line", "LocationName", "Destination"])["BRD_Time"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

arrivals_df = arrivals_df.dropna(subset=["GapMin"])

st.title("WMATA Rail Punctuality Analysis")

st_col1, st_col2 = st.columns(2)

with st_col1:
    hist_fig = px.histogram(
        filtered_df,
        x="Diff",
        nbins=int(filtered_df["Diff"].max() - filtered_df["Diff"].min()) + 1,
        title="Distribution of Prediction Error - " + title_color + " - " + title_location,
        labels={"Diff": "Actual Arrival minus Predicted Arrival (minutes)", "count": "Count"},
        color_discrete_sequence=[plot_color],
        histnorm="probability"
    )
    hist_fig.update_layout(bargap=0.05,height=350, margin=dict(t=30, b=30),yaxis_title="Proportion")
    st.plotly_chart(hist_fig, use_container_width=True)

gap_filtered = arrivals_df.copy()
if selected_line != "All":
    gap_filtered = gap_filtered[gap_filtered["Line"] == selected_line]
if selected_location != "All":
    gap_filtered = gap_filtered[gap_filtered["LocationName"] == selected_location]

with st_col2:
    gap_fig = px.histogram(
        gap_filtered,
        x="GapMin",
        nbins=30,
        title="Time Between Train Arrivals - " + title_color + " - " + title_location,
        labels={"GapMin": "Time Between Trains (minutes)"},
        color_discrete_sequence=[plot_color],
        histnorm="probability"
    )
    gap_fig.update_layout(
        bargap=0.05,
        height=350,
        margin=dict(t=30, b=30),
        xaxis_title="Gap (minutes)",
        yaxis_title="Proportion"
    )
    st.plotly_chart(gap_fig, use_container_width=True)

avg_diff = filtered_df.groupby("Min", as_index=False)["Diff"].mean()

box_fig = px.box(
    filtered_df,
    x="Min",
    y="Diff",
    points="outliers",
    title= "Prediction Error Box Plots by Estimated Minutes Out - " + title_color + " - " + title_location,
    labels={"Min": "Estimated Minutes Out", "Diff": "Prediction Error (minutes)"},
    color_discrete_sequence=[plot_color]
)

box_fig.update_yaxes(
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor="black"
)
box_fig.update_layout(height=350, margin=dict(t=30, b=30))
st.plotly_chart(box_fig, use_container_width=True)



Overwriting streamlit_app.py
